# pm4py-ucm — dashboards and exports

The other tutorials turn an event log into **models** (UCMs, families,
scenarios). This one turns the same log into **dashboards**: small,
composable *widgets* — KPIs, breakdowns, targets — computed over the log
and packaged as a single **self-contained HTML file** you can email,
archive, or open offline.

Two ideas make the dashboard layer unusual:

* **The app *is* the export.** The interactive Dashboards view in the web
  app and the downloadable HTML file are the *same artifact* built by the
  same function (`dashboard_html`). There is no separate "report
  generator" to drift out of sync.
* **One engine, two languages.** Every widget value is computed by an
  engine that exists twice — in Python (`pm4py_ucm.algo.dashboards.engine`,
  which this notebook calls) and in JavaScript (for the browser and the
  offline HTML). A parity test pins them together, so a number you compute
  here in Python is byte-for-byte the number the exported page shows.

You will:

1. build the **fact table** — the compact per-case snapshot every widget
   reads;
2. browse the **metric catalog** and compute individual **widgets**
   (KPIs, segmented breakdowns, tables);
3. add **filters**, **targets** and a **scorecard**;
4. write your own metrics in the **ƒ custom-formula language**;
5. assemble a dashboard and **export** it as one self-contained HTML file;
6. see the **related exports** — embedding a mined model, and the
   multi-section session report.

The example log is the bundled `ClaimsPaymentLog` (5 600 cases, 78 126
events) — an *interval* log (each event has a start and a completion
timestamp), so activity service and waiting times are available.

## 1. Setup and the fact table

In [1]:
import sys, zipfile
from pathlib import Path

# Import the repo's pm4py_ucm even without `pip install -e .`.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "pm4py_ucm").is_dir() and (REPO_ROOT.parent / "pm4py_ucm").is_dir():
    REPO_ROOT = REPO_ROOT.parent          # running from demo/
sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import pm4py
import pm4py_ucm

DEMO = REPO_ROOT / "demo"
log_path = DEMO / "ClaimsPaymentLog.xes"
if not log_path.exists():
    with zipfile.ZipFile(DEMO / "ClaimsPaymentLog.zip") as zf:
        zf.extractall(DEMO)

df = pm4py.read_xes(str(log_path))
print(f"{df['case:concept:name'].nunique():,} cases, {len(df):,} events")

C:\Users\jucmn\AppData\Roaming\Python\Python313\site-packages\pm4py\utils.py:1005: UserWarning: In the current version, the import/export operation uses `r4pm` by default for importing/exporting files faster.
  warnings.warn(


5,600 cases, 78,126 events


`build_fact_table` reduces the log to a **columnar per-case snapshot**:
per-case start/end times, a CSR (compressed-sparse-row) index of each
case's activity sequence, the activity/resource dictionaries, and the
**case-constant attributes** it detects (the same detection the model
families use). Everything a widget needs is here; nothing else is kept.
It is deliberately small and JSON-serialisable — that is what lets the
whole thing ship to a browser and recompute offline.

In [2]:
from pm4py_ucm.algo.dashboards import build_fact_table

table = build_fact_table(df, log_name="ClaimsPaymentLog")

print(f"cases          : {table.n_cases:,}")
print(f"events         : {table.n_events:,}")
print(f"activities     : {len(table.activities)}")
print(f"interval log   : {table.interval_log}   (service/waiting times available)")
print(f"attributes     :")
for a in table.attributes:
    print(f"    {a.name:<14} {a.type}")

cases          : 5,600
events         : 78,126
activities     : 25
interval log   : True   (service/waiting times available)
attributes     :
    Broker         enumeration
    Channel        enumeration
    Claim_Value    integer
    Country        enumeration
    Product_Group  enumeration


## 2. The metric catalog

A widget names a metric from the **catalog** — process-level (case
duration, event count, rework, time-between activities, WIP / arrival /
completion rates), activity-level (frequency, presence, sojourn, service,
waiting), and edge-level (directly-follows frequency, time, branch share).
Metrics the log can't support (service time on a log without start
timestamps) are simply absent — never fabricated.

In [3]:
from pm4py_ucm.algo.dashboards import CATALOG

pd.DataFrame([
    {"id": m.id, "level": m.level, "result": m.result_type, "what": m.label}
    for m in CATALOG
])

,id,level,result,what
0,duration,process,time,Case duration
1,timeBetween,process,time,Time between two activities
2,wip,process,rate,Work in progress (WIP)
3,arrivalRate,process,rate,Arrival rate
4,completionRate,process,rate,Completion rate
5,eventCount,process,count,Events per case
6,rework,process,percent,Rework rate
7,actFreq,activity,count,Activity frequency
8,actPresence,activity,percent,Activity presence
9,actRepeats,activity,count,Repeats per case


## 3. Computing a widget

`compute_widget(spec, table)` takes a **widget spec** (a plain dict) and
returns *data* — a value, its formatted text, per-segment breakdowns,
case counts, target state. This is exactly the structure the app and the
exported HTML render, so what you compute here is what a reader sees.

The simplest widget is a KPI — one metric, one aggregation:

In [4]:
from pm4py_ucm.algo.dashboards import compute_widget

kpi = compute_widget(
    {"id": "dur", "metric": "duration", "viz": "kpi", "agg": "median"},
    table,
)
print(f"{kpi['title']}: {kpi['text']}   (over {kpi['nCases']:,} cases)")

Case duration: 15.2 d   (over 5,600 cases)


Note the unit: durations are reported in the largest **legible** unit, so
a short value reads `2.4 h` or `43 m` rather than `0.0 d`. The same
`fmt(value, unit)` drives every cell in the app and the export.

## 4. Filters and segmentation

A `filter` narrows which cases a widget measures; a `segment` breaks the
result down along one axis (→ a **bar**) or two (→ a **table**). Axes are
`resource`, `variant`, a time bucket (`month`, `quarter`, `weekday`, …),
or a case attribute as `attr:<name>`. A filter can also be a **date
range** — `{"field": "date", "value": [from, to]}`, either end open.

In [5]:
# Average case duration per country (one axis → a bar chart's `series`).
by_country = compute_widget(
    {"id": "dur_country", "metric": "duration", "viz": "bar", "agg": "avg",
     "segment": {"rows": "attr:Country"}},
    table,
)
pd.DataFrame([
    {"country": s["label"], "avg duration": s["text"], "cases": s["nCases"]}
    for s in by_country["series"]
])

,country,avg duration,cases
0,AUS,14.8 d,1950
1,IDN,14.8 d,672
2,MYS,14.9 d,520
3,NZL,15.1 d,1769
4,SGP,14.9 d,689


In [6]:
# Events by Country x Product_Group (two axes → a table's `rows`/`cols`/`cells`).
# Add a dashboard-level filter: only high-value claims.
tbl = compute_widget(
    {"id": "ev", "metric": "eventCount", "viz": "table", "agg": "sum",
     "segment": {"rows": "attr:Country", "cols": "attr:Product_Group"}},
    table,
    dashboard_filters=[{"field": "attr:Claim_Value", "op": ">", "value": 5000}],
)
pd.DataFrame(
    [[c["value"] for c in row] for row in tbl["cells"]],
    index=pd.Index(tbl["rows"], name="Country"),
    columns=tbl["cols"],
)

,Boat,Content,Health,Motor,Pet,Property,Travel
Country,,,,,,,
AUS,649.0,2425.0,6262.0,3708.0,292.0,3219.0,1626.0
IDN,194.0,944.0,1974.0,1403.0,77.0,1254.0,562.0
MYS,157.0,524.0,1613.0,1070.0,58.0,1098.0,435.0
NZL,507.0,2386.0,5709.0,3600.0,335.0,3019.0,1448.0
SGP,155.0,900.0,2126.0,1605.0,149.0,918.0,815.0


`dashboard_filters` stack **on top of** each widget's own filter, so a
reader-side filter narrows every widget at once without editing any of
them — the same mechanism the exported page's filter bar uses.

## 5. Visualisations

A widget's `viz` says how its figure is **drawn**. Most shapes are a
rendering choice rather than a different computation, and the composer
offers only the ones that say something true about the metric and the
segmentation:

| Segmentation | `viz` |
| --- | --- |
| no axes | `kpi`, `gauge` (needs a target), `hist`, `box` |
| one axis | `bar`; `line` on a time axis; `pie` when the aggregation is a `sum` |
| two axes | `table` |

Two of them return **extra data** you can read from Python: `hist` bins the
per-case values, and `box` returns a five-number summary with Tukey
whiskers and outliers. `line`, `gauge` and `pie` re-draw figures the engine
already returns (`series`, `value`), so they add no new numbers.

In [7]:
# A histogram bins the per-case values; a box plot summarises them.
h = compute_widget(
    {"id": "dur_hist", "metric": "duration", "viz": "hist", "agg": "avg"},
    table,
)["hist"]
print(f"histogram: {len(h['bins'])} bins over "
      f"{h['min']:.2f}–{h['max']:.1f} d ({h['n']:,} cases)")

b = compute_widget(
    {"id": "dur_box", "metric": "duration", "viz": "box", "agg": "median"},
    table,
)["box"]
print(f"box plot:  min {b['min']:.2f} | Q1 {b['q1']:.1f} | "
      f"median {b['median']:.1f} | Q3 {b['q3']:.1f} | max {b['max']:.1f} d"
      f"   ({b['nOutliers']} outliers)")

histogram: 20 bins over 0.04–32.0 d (5,600 cases)
box plot:  min 0.04 | Q1 7.1 | median 15.2 | Q3 21.2 | max 32.0 d   (0 outliers)


A pie only says something when its slices **add up to a whole**, so it is
offered for a `sum` and not for an average — "average duration by country"
sums to nothing. `sum` is available for times as well as counts, which
makes a total duration ("case-days by country") pie-able:

In [8]:
# `sum` over a time metric: a total, not an average — what a pie needs.
days_by_country = compute_widget(
    {"id": "days_country", "metric": "duration", "viz": "pie", "agg": "sum",
     "segment": {"rows": "attr:Country"}},
    table,
)
total = sum(s["value"] for s in days_by_country["series"])
pd.DataFrame([
    {"country": s["label"], "total": s["text"],
     "share": f"{100 * s['value'] / total:.1f}%", "cases": s["nCases"]}
    for s in sorted(days_by_country["series"],
                    key=lambda s: -s["value"])
])

,country,total,share,cases
0,AUS,"28,928 d",34.6%,1950
1,NZL,"26,677 d",31.9%,1769
2,SGP,"10,243 d",12.3%,689
3,IDN,"9,966 d",11.9%,672
4,MYS,"7,748 d",9.3%,520


## 6. Targets and the scorecard

A widget can carry a **target** — a goal, a warning threshold, and a
direction. `compute_widget` then returns a traffic-light `state`
(`met` / `risk` / `missed`). A **scorecard** collects the state of every
targeted widget into one at-a-glance table. In the app a segmented
target's scorecard row expands to the segments that **breached** it, and
clicking one filters the dashboard to that segment.

In [9]:
from pm4py_ucm.algo.dashboards import scorecard

specs = [
    {"id": "median_dur", "metric": "duration", "viz": "kpi", "agg": "median",
     "target": {"on": True, "dir": "<=", "value": 14, "warn": 21}},
    {"id": "rework", "metric": "rework", "viz": "kpi",
     "target": {"on": True, "dir": "<=", "value": 20, "warn": 35}},
]
for s in specs:
    w = compute_widget(s, table)
    print(f"{w['title']:<16} {w['text']:>8}   {w['state']:<7} ({w.get('sub','')})")

print("\nScorecard:")
pd.DataFrame(scorecard(specs, table))[["title", "goal", "actual", "state"]]

Case duration      15.2 d   risk    (target ≤ 14 d)
Rework rate         35.5%   missed  (target ≤ 20%)

Scorecard:


,title,goal,actual,state
0,Case duration,≤ 14 d,15.2 d,risk
1,Rework rate,≤ 20%,35.5%,missed


## 7. The ƒ custom-formula language

When the catalog does not name what you want to measure, write it as a
per-case **formula**. The grammar is tiny and closed — parsed to an
explicit AST, never `eval`'d — and every expression evaluates, per case,
to a **number or null**:

| function | value (per case) |
|---|---|
| `duration()` | case length, in days |
| `contains(act)` | `1` if the case has `act`, else `0` |
| `count(act)` | occurrences of `act` |
| `time_between(a, b)` | days, first `a` → first `b` after it; `null` if never |
| `timestamp(act)` | epoch seconds of the first `act`, else `null` |
| `attr(name)` | a **numeric** case attribute, else `null` |
| `attr(name) == "value"` | `1` where the categorical attribute equals `value` (use `!=`, or `or`-chain for a set), else `0`/`null` |

A quoted value is a value only in that categorical equality — `attr("Channel") == "Web"` — matched against the attribute's dictionary; everywhere else strings are just the names inside a call. With `+ - * /`, comparisons, `and` / `or` / `not`, parentheses, and an
optional trailing `where <predicate>`. The **result type** is inferred:
a comparison or a bare `contains()` → a 0/1 indicator (`percent`); any
time function → `time`; otherwise `count`. That fixes the unit and the
aggregations offered — a formula behaves exactly like a catalog metric.

In [10]:
from pm4py_ucm.algo.dashboards import compile_formula, FormulaError

acts = tuple(table.activities)
attrs = tuple(a.name for a in table.attributes)

examples = [
    'count("Analyze Claim") > 1',                     # reworked cases (%)
    'duration() where attr("Claim_Value") > 5000',    # duration of big claims
    'time_between("Register Claim", "Schedule Payment")',
    'contains("Cancel Claim")',                        # cancelled cases (%)
]
for f in examples:
    compiled = compile_formula(f, activities=acts, attributes=attrs)
    print(f"{f!r:<52} -> {compiled['resultType']}")

# A broken formula raises FormulaError (unknown function, bad arity, ...).
try:
    compile_formula('median()', activities=acts, attributes=attrs)
except FormulaError as e:
    print("\nrejected:", e)

'count("Analyze Claim") > 1'                         -> percent
'duration() where attr("Claim_Value") > 5000'        -> time
'time_between("Register Claim", "Schedule Payment")' -> time
'contains("Cancel Claim")'                           -> percent


In [11]:
# Use a formula in a widget: metric "custom", the text in params.formula.
rework_share = compute_widget(
    {"id": "fx1", "metric": "custom", "viz": "kpi",
     "params": {"formula": 'count("Analyze Claim") > 1'}},
    table,
)
big_claim_dur = compute_widget(
    {"id": "fx2", "metric": "custom", "viz": "kpi", "agg": "median",
     "params": {"formula": 'duration() where attr("Claim_Value") > 5000'}},
    table,
)
print(f"cases reanalysed (>1 Analyze Claim): {rework_share['text']}")
print(f"median duration, claims over 5000  : {big_claim_dur['text']}")

cases reanalysed (>1 Analyze Claim): 1.9%
median duration, claims over 5000  : 15.8 d


## 8. Assemble a dashboard and export it

A dashboard is just an **ordered list of specs** plus an optional
dashboard-level filter. `write_dashboard` (or `dashboard_html`, which
returns the string) packs the fact table, the widgets, and the whole
engine into **one self-contained HTML file** — no server, no external
assets, works offline. Passing `read_only=True` produces the *reader*
form: filters, drill-down, CSV export and the scorecard all still work;
only editing the widget set is disabled.

`specs` seeds the dashboard. In the app the reader takes over from
there: widgets are dragged to reorder and resized from a corner, a
log can hold several **named dashboards** (switched from the header),
and each is exportable on its own, **all in one file**, or saved as a
small JSON *definition* that reloads here or onto another log.

In [12]:
from pm4py_ucm.algo.dashboards import write_dashboard

OUT = DEMO / "output" / "dashboards"
OUT.mkdir(parents=True, exist_ok=True)

dashboard = [
    {"id": "w1", "title": "Median case duration", "metric": "duration",
     "viz": "kpi", "agg": "median",
     "target": {"on": True, "dir": "<=", "value": 14, "warn": 21}},
    {"id": "w2", "title": "Duration by country", "metric": "duration",
     "viz": "bar", "agg": "avg", "segment": {"rows": "attr:Country"}},
    {"id": "w3", "title": "Events by country x product", "metric": "eventCount",
     "viz": "table", "agg": "sum",
     "segment": {"rows": "attr:Country", "cols": "attr:Product_Group"}},
    {"id": "w4", "title": "Reanalysed claims", "metric": "custom", "viz": "kpi",
     "params": {"formula": 'count("Analyze Claim") > 1'}},
]

html_path = write_dashboard(
    str(OUT / "claims_dashboard.html"), table,
    name="Claims operations", specs=dashboard, read_only=True,
)
size_mb = Path(html_path).stat().st_size / 1e6
print(f"{html_path}  ({size_mb:.1f} MB, self-contained — open it in a browser)")

C:\Users\jucmn\Claude\pm4py-ucm\.claude\worktrees\pedantic-morse-cba059\demo\output\dashboards\claims_dashboard.html  (1.4 MB, self-contained — open it in a browser)


Open the file in any browser: it is the same interactive dashboard the
web app shows, driven by the JavaScript twin of the engine you just
called. Because the values were computed by one engine, the page's
numbers match this notebook's exactly.

## 9. Related exports

**Embed a mined model.** A widget with `viz: "model"` shows a process
model; the model itself is rendered server-side (the browser can't mine)
and handed over as inline SVG via `model_svg`, so it stays navigable in
the export. Here we mine the log and pass both notations:

In [13]:
from pm4py_ucm.visualization.ucm import svg as ucm_svg

ucm = pm4py_ucm.discover_ucm_inductive(df, decomposition="auto")
model_svg = {
    "ucm":  ucm_svg.model_to_svg(ucm, "ucm"),
    "bpmn": ucm_svg.model_to_svg(ucm, "bpmn"),
}
with_model = write_dashboard(
    str(OUT / "claims_with_model.html"), table,
    name="Claims operations", specs=dashboard + [
        {"id": "m", "title": "Mined model", "metric": "duration", "viz": "model"},
    ],
    model_svg=model_svg, read_only=True,
)
print(f"wrote {Path(with_model).name} "
      f"({Path(with_model).stat().st_size/1e6:.1f} MB, model embedded as vector SVG)")

wrote claims_with_model.html (1.6 MB, model embedded as vector SVG)


**The session report.** The web app's Dashboards view can also emit a
multi-section **session report** — a table of contents, a scorecard, one
section per dashboard, the process model, and (when a family has been
mined) the family statistics report folded in. That report is assembled
in the browser from the live page (so it carries the reader's actual
widgets); the same self-contained-HTML machinery underpins it.

Everything in this notebook is also point-and-click in the **web app**
(`web/streamlit_app_v5.py`, the **Dashboards** view): build widgets from
the catalog, write ƒ formulas with a live validity check, pin a mined
model, and download the dashboard or the session report. See
[`docs/dashboards.md`](../docs/dashboards.md) for the semantic contract
(every metric's exact definition and the engine's rounding / weighting
decisions).

## 10. Wrap-up

```python
from pm4py_ucm.algo.dashboards import (
    build_fact_table, compute_widget, scorecard,
    compile_formula, dashboard_html, write_dashboard,
)
table = build_fact_table(df, log_name="…")     # per-case snapshot
compute_widget(spec, table)                     # one widget's data
scorecard(specs, table)                         # targeted widgets' states
compile_formula('duration() where attr("x") > 0', activities=…, attributes=…)
write_dashboard("dash.html", table, specs=[…], read_only=True)   # self-contained export
```

Where to go next:

* [`docs/dashboards.md`](../docs/dashboards.md) — the metric catalog with
  exact definitions, the ƒ grammar, and the engine's design decisions.
* `model_families_tutorial.ipynb` — the family statistics report that the
  session report can fold in.
* The **Dashboards** view of `web/streamlit_app_v5.py` — all of the above
  interactively, plus drag-to-arrange, several named dashboards, the
  breach drill-down, and the session report download.